In [ ]:
import json
import pandas as pd
from pathlib import Path

In [ ]:
results_dir = Path("../data/benchmark_results")
result_files = sorted(results_dir.glob("training_timing_benchmark_seed*.json"))
assert len(result_files) > 0, f"No benchmark results found in {results_dir}"

result_path = result_files[-1]
print(f"Loading: {result_path.name}")

with open(result_path, 'r') as f:
    data = json.load(f)

print(f"Seed: {data['seed']}")
print(f"Dataset: {data['dataset_name']}")
print(f"out_dim: {data['out_dim']}, batch_size: {data['batch_size']}")
print(f"Warmup batches: {data['num_warmup_batches']}, Timed batches: {data['num_timed_batches']}")
print(f"Number of configs: {len(data['results'])}")

In [ ]:
import re

def parse_config(label):
    m_s = re.search(r'_s(\d+)$', label)
    num_prior_samples = int(m_s.group(1)) if m_s else "-"

    if '_aat_shared_' in label:
        cov_type = 'aat_shared'
    elif '_aat_' in label:
        cov_type = 'aat'
    elif '_diag_' in label:
        cov_type = 'diag'
    else:
        cov_type = "-"

    m_r = re.search(r'_r(\d+)', label)
    if m_r:
        cov_rank = int(m_r.group(1))
    elif cov_type == 'diag':
        cov_rank = 0
    else:
        cov_rank = "-"

    return cov_type, cov_rank, num_prior_samples

rows = []
for label, r in data["results"].items():
    if r.get("oom"):
        print(f"skipping {label}: did not fit on this GPU ({r.get('error', 'OOM')})")
        continue
    cov_type, cov_rank, num_prior_samples = parse_config(label)
    rows.append({
        "Config": label,
        "Model": r["model_name"],
        "cov_type": cov_type,
        "cov_rank": cov_rank,
        "num_prior_samples": num_prior_samples,
        "l2_weight": r.get("l2_weight", 0),
        "Mean batch time (ms)": r["mean_batch_sec"] * 1000,
        "Std batch time (ms)": r["std_batch_sec"] * 1000,
        "Median batch time (ms)": r["median_batch_sec"] * 1000,
    })

df = pd.DataFrame(rows).set_index("Config")

In [ ]:
df[["Model", "cov_type", "cov_rank", "num_prior_samples", "l2_weight",
    "Mean batch time (ms)", "Std batch time (ms)", "Median batch time (ms)"]]

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['DejaVu Serif']

ranks = [4, 8, 16, 64]

aat_s512 = df[(df["cov_type"] == "aat") & (df["num_prior_samples"] == 512)].sort_values("cov_rank")
aat_shared_s512 = df[(df["cov_type"] == "aat_shared") & (df["num_prior_samples"] == 512)].sort_values("cov_rank")

mnl = df.loc["MNL", "Mean batch time (ms)"]
mxl_diag = df.loc["MXL_diag_l20.0001_s512", "Mean batch time (ms)"]

fig, ax = plt.subplots(figsize=(4, 3.5))

ax.plot(aat_s512["cov_rank"].values, aat_s512["Mean batch time (ms)"].values, 'o-', color='tab:blue')
ax.plot(aat_shared_s512["cov_rank"].values, aat_shared_s512["Mean batch time (ms)"].values, 's-', color='tab:orange')
ax.axhline(mnl, ls='--', color='tab:green')
ax.axhline(mxl_diag, ls='--', color='tab:red')

ax.set_xlabel('Rank (r)', fontsize=13)
ax.set_ylabel('Mean Time (ms)', fontsize=13)
ax.set_xticks(ranks)
ax.set_xticklabels([str(r) for r in ranks])
ax.set_yticks([0, 10, 20, 30, 40, 50, 60, 70, 80, 90])
ax.tick_params(axis='y', labelsize=9)
ax.tick_params(axis='x', labelsize=10)

y_top = ax.get_ylim()[1]

band_pad = y_top * 0.035
band_bottom, band_top = min(mnl, mxl_diag) - band_pad, max(mnl, mxl_diag) + band_pad
trans = ax.get_yaxis_transform()
ax.add_patch(FancyBboxPatch((1.0, band_bottom), 0.56, band_top - band_bottom,
                            boxstyle='round,pad=0.0', facecolor='lightgray', alpha=0.35,
                            edgecolor='none', clip_on=False, transform=trans))
ax.text(1.035, band_bottom - band_pad * 0.6, 'Constant\nno Cov. Matrix',
        transform=trans, ha='left', va='top', fontsize=8, fontstyle='italic',
        color='dimgray', clip_on=False)

labels = [
    ('MXL-\nPers-Cov', aat_s512["Mean batch time (ms)"].values[-1], 'tab:blue', 2),
    ('MXL-\nShared-Cov', aat_shared_s512["Mean batch time (ms)"].values[-1], 'tab:orange', 2),
    ('MXL', mxl_diag, 'tab:red', 1),
    ('MNL', mnl, 'tab:green', 1),
]
line_h = y_top * 0.062
placed = sorted(labels, key=lambda t: t[1])
ys = []
for text, y, color, nlines in placed:
    need = line_h * nlines
    if ys and y - need / 2 < ys[-1][1] + ys[-1][0] / 2:
        y = ys[-1][1] + ys[-1][0] / 2 + need / 2
    ys.append((need, y))
for (text, _, color, nlines), (_, y) in zip(placed, ys):
    ax.annotate(text, xy=(1, y), xytext=(4, 0), color=color,
                fontsize=11, fontweight='bold', va='center', clip_on=False,
                xycoords=('axes fraction', 'data'), textcoords='offset points')

fig.subplots_adjust(left=0.14, right=0.62)
ax.grid(True, alpha=0.3)
plt.savefig('../data/plots/training_timing_plot.pdf', bbox_inches='tight')
plt.show()